# NJ Congressional Redistricting (Team LL)

In [1]:
!pip install geojson
!pip install shapely
!pip install PyShp
!pip install networkx

In [2]:
import geojson
import pandas as pd
import numpy as np
import networkx as nx
import time
import csv
import ast
import math
import random
import shapefile as shp
from shapely.geometry import Polygon,shape,MultiPolygon
import shapely.ops
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)
warnings.filterwarnings('ignore', message="The 'type' attribute is deprecated")

### Data

In [3]:
nj_current_assignment = pd.read_csv('Map_Data/precinct-assignments-congress-nj.csv', dtype={'GEOID20': str})
nj_precinct_data = pd.read_csv('Map_Data/precinct-data-congress-nj.csv', dtype={'GEOID20': str})
nj_contiguity = pd.read_csv('Contiguity_nj.csv', header=None, dtype={0: str})
nj_contiguity.columns = ['Precinct','Neighbors']

shpfile = 'Map_Data/nj_vtd_2020_bound/nj_vtd_2020_bound.shp'
dbffile = 'Map_Data/nj_vtd_2020_bound/nj_vtd_2020_bound.dbf'
shxfile = 'Map_Data/nj_vtd_2020_bound/nj_vtd_2020_bound.shx'

shpfile = shp.Reader(shp=shpfile, shx=shxfile, dbf=dbffile)
nj_precinct_boundaries={}
for sr in shpfile.iterShapeRecords():
    geom = sr.shape # get geo bit
    rec = sr.record # get db fields
    nj_precinct_boundaries[rec[3]]=geom

print(len(nj_precinct_data), 'precincts')

6361 precincts


### Helper functions (from the course starter notebook)

In [4]:
def isDistrictContiguous(district_num, assignment, contiguity_list, print_isolates=False, ignore_list=[]):
    ## input:
    ## district_num: the district number
    ## assignment: the assignment from precinct to district
    ## contiguity_list: the list of neighbors for each precinct, from the csv file
    contiguity_list.columns = ['Precinct','Neighbors']
    district_graph = nx.Graph() #creates an empty undirected graph
    district_nodes = assignment[assignment['District']==district_num]['GEOID20'].tolist()
    for i in ignore_list:
        try:
            district_nodes.remove(i)
        except ValueError:
            pass
    district_graph.add_nodes_from(district_nodes)
    for id in district_nodes:
        neighbors = ast.literal_eval(contiguity_list[contiguity_list['Precinct']==id]['Neighbors'].values.tolist()[0])
        # needed to convert string to list because the csv encodes the list as a string
        for neighbor in neighbors:
            if neighbor in district_nodes:
                district_graph.add_edge(id,neighbor)
    if(print_isolates):
        print(list(nx.isolates(district_graph)))
    return nx.is_connected(district_graph)

def getDistrictPopulations(assignment,data_file, num_district):
    population = {}
    for i in range (1,num_district+1):
        population[i] = data_file[data_file['GEOID20'].isin(assignment[assignment['District']==i]['GEOID20'])]['Total_2020_Total'].sum()
    return population

def getDistrictShape(district_id, assignment, boundaries):
    list_precincts = assignment[assignment['District']==district_id]['GEOID20']
    precinct_shapes = []
    for i in list_precincts:
        if shape(boundaries[i]).type == 'Polygon':
            precinct_shapes.append(Polygon(shape(boundaries[i])))
        elif shape(boundaries[i]).type == 'MultiPolygon':
            precinct_shapes.append(MultiPolygon(shape(boundaries[i])))
    district_shape = shapely.ops.unary_union(precinct_shapes)
    return district_shape

def pp_compactness(geom): # Polsby-Popper
    p = geom.length
    a = geom.area
    return (4*np.pi*a)/(p*p)

def box_reock_compactness(geom): # Reock on a rectangle bounding box
    a = geom.area
    bb = geom.bounds # bounds gives you the minimum bounding box (rectangle)
    bba = abs(bb[0]-bb[2])*abs(bb[1]-bb[3])
    return a/bba

# Unfair Map (Safa Karagoz)

**Goal:** draw a map that favors Republicans. NJ votes about 56% Democratic (DRA's 2016-2021 composite), and the current map elects 9 Democrats and 3 Republicans.

**Constraints:** every district is contiguous and within 2% of the ideal population (the assignment allows 10%, prefers 5%).

**Algorithm:**
1. **Starting map:** recursive spanning-tree partitioning. Take a random spanning tree of the precinct graph and cut the edge that separates one district's worth of population. Repeat 11 times. Each district is a connected subtree, so it is contiguous.
2. **Phase 1, maximize Republican seats:** short bursts of ReCom moves. A ReCom move merges two neighboring districts and re-splits them along a new random spanning tree, keeping both within 2% of ideal population. We run short bursts of moves from the best map found so far and keep any map that scores at least as well. The score is the number of districts with a Republican share of at least 52% (a margin so the seats are not coin flips), with ties broken by the Republican share of the next most competitive district.
3. **Phase 2, hide the gerrymander:** same short bursts, but now a map is only kept if it still has all the safe Republican seats. The current map has one majority-Black district, and removing it would invite a Voting Rights Act lawsuit, so until some district is over 50% Black by voting-age population the score is just the highest Black share. After that, the score is the number of district boundary edges (fewer means more compact) plus 10 for every piece a county is split into. In this phase the spanning trees are county-aware: edges inside a county are always added before edges that cross a county line, so the new district lines tend to follow county lines.

**Measuring unfairness:** seats won versus statewide vote share, efficiency gap, and mean-median difference, all on the composite election data. Spanning-tree splits keep districts compact, so we also compare compactness and county splits with the current map to check the map is not an obvious gerrymander. Finally, we count seats under each individual election (2016-2021) to see how safe the Republican seats are.

In [5]:
K = 12                 # NJ congressional districts
SEED = 3
N_BURSTS = 2500       # phase 1
N_BURSTS_2 = 2500     # phase 2
BURST_LEN = 10
COUNTY_WEIGHT = 10     # phase 2: one extra county piece costs as much as this many boundary edges
R_THRESHOLD = 0.52     # a district counts as a Republican seat at 52% or more
POP_TOL = 0.02         # max population deviation from ideal
BLACK_TARGET = 0.50    # phase 2: keep one district majority-Black by voting-age population (Voting Rights Act)

pop = dict(zip(nj_precinct_data['GEOID20'], nj_precinct_data['Total_2020_Total']))
dem = dict(zip(nj_precinct_data['GEOID20'], nj_precinct_data['Dem_2016-2021_Comp']))
rep = dict(zip(nj_precinct_data['GEOID20'], nj_precinct_data['Rep_2016-2021_Comp']))
black_vap = dict(zip(nj_precinct_data['GEOID20'], nj_precinct_data['Black_2020_VAP']))
vap = dict(zip(nj_precinct_data['GEOID20'], nj_precinct_data['Total_2020_VAP']))

# adjacency from the precomputed contiguity index (made symmetric)
adj = {p: set() for p in nj_contiguity['Precinct']}
for p, n in zip(nj_contiguity['Precinct'], nj_contiguity['Neighbors']):
    for q in ast.literal_eval(n):
        adj[p].add(q)
        adj[q].add(p)

# Water-only precincts (GEOID ends in ZZZZZZ) have no people. We leave them out while building
# districts so two pieces of land can't count as connected across open water, then attach them at the end.
water = sorted(p for p in adj if p.endswith('ZZZZZZ'))
land = sorted(p for p in adj if p not in water)
land_adj = {p: sorted(q for q in adj[p] if q not in water) for p in land}

IDEAL = sum(pop.values()) / K
rng = random.Random(SEED)
COUNTY_AWARE = False   # turned on in phase 2

print('water precincts:', water)
print('land graph connected:', nx.is_connected(nx.Graph([(p, q) for p in land for q in land_adj[p]])))
print('ideal district population:', round(IDEAL))

water precincts: ['34001ZZZZZZ', '34009ZZZZZZ', '34011ZZZZZZ', '34025ZZZZZZ', '34029ZZZZZZ']
land graph connected: True
ideal district population: 774083


In [6]:
def random_spanning_tree(nodes):
    # Kruskal's algorithm with random edge weights. When COUNTY_AWARE is on, edges crossing a county line
    # get +1 so every within-county edge comes first (the first 5 GEOID digits are state + county).
    edges = [(u, v) for u in sorted(nodes) for v in land_adj[u] if u < v and v in nodes]
    edges.sort(key=lambda e: rng.random() + (1.0 if COUNTY_AWARE and e[0][:5] != e[1][:5] else 0.0))
    parent = {u: u for u in nodes}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    tree = {u: [] for u in nodes}
    needed = len(nodes) - 1
    for u, v in edges:
        ru, rv = find(u), find(v)
        if ru != rv:
            parent[ru] = rv
            tree[u].append(v)
            tree[v].append(u)
            needed -= 1
            if needed == 0:
                break
    return tree


def balanced_cut(tree, nodes, target_a, tol, total, target_b=None):
    # Find tree edges whose removal leaves a subtree with population target_a +- tol
    # (and the rest target_b +- tol if given). Returns the precincts of one random such subtree.
    root = min(nodes)
    parent = {root: None}
    order = [root]
    stack = [root]
    while stack:
        u = stack.pop()
        for v in tree[u]:
            if v not in parent:
                parent[v] = u
                order.append(v)
                stack.append(v)

    subtree_pop = dict.fromkeys(nodes, 0)
    for u in reversed(order):
        subtree_pop[u] += pop[u]
        if parent[u] is not None:
            subtree_pop[parent[u]] += subtree_pop[u]

    candidates = [u for u in order[1:]
                  if abs(subtree_pop[u] - target_a) <= tol
                  and (target_b is None or abs(total - subtree_pop[u] - target_b) <= tol)]
    if not candidates:
        return None

    u = rng.choice(candidates)
    side = {u}
    stack = [u]
    while stack:
        x = stack.pop()
        for y in tree[x]:
            if parent[y] == x:
                side.add(y)
                stack.append(y)
    return side

In [7]:
def initial_partition(tol_frac=0.01):
    # carve off one district at a time; the target is recomputed so leftover population stays balanced
    remaining = set(land)
    districts = {}
    for k in range(K - 1):
        remaining_pop = sum(pop[n] for n in remaining)
        target = remaining_pop / (K - k)
        for attempt in range(2000):
            side = balanced_cut(random_spanning_tree(remaining), remaining, target, tol_frac * IDEAL, remaining_pop)
            if side:
                break
        else:
            raise RuntimeError('could not carve district ' + str(k))
        districts[k] = side
        remaining -= side
    districts[K - 1] = remaining
    return districts


def rep_shares(districts):
    return {k: math.fsum(rep[n] for n in s) / math.fsum(dem[n] + rep[n] for n in s) for k, s in districts.items()}


def score(shares):
    s = sorted(shares.values(), reverse=True)
    seats = sum(x >= R_THRESHOLD for x in s)
    next_closest = s[seats] if seats < K else 0
    return seats + next_closest


start = time.time()
initial = initial_partition()
shares = rep_shares(initial)
print(f'initial map built in {time.time() - start:.1f}s')
print('max population deviation: %.2f%%' % (max(abs(sum(pop[n] for n in s) / IDEAL - 1) for s in initial.values()) * 100))
print('Republican seats (>50%):', sum(x > 0.5 for x in shares.values()), '| score:', round(score(shares), 3))

initial map built in 0.2s
max population deviation: 0.99%
Republican seats (>50%): 5 | score: 5.415


In [8]:
def recom_step(districts, assignment):
    # pick a random edge between two districts, merge those districts, re-split along a random spanning tree
    cut_edges = [(u, v) for u in land for v in land_adj[u] if u < v and assignment[u] != assignment[v]]
    u, v = rng.choice(cut_edges)
    a, b = assignment[u], assignment[v]
    merged = districts[a] | districts[b]
    total = sum(pop[n] for n in merged)
    for attempt in range(20):
        side = balanced_cut(random_spanning_tree(merged), merged, IDEAL, POP_TOL * IDEAL, total, IDEAL)
        if side:
            districts[a] = side
            districts[b] = merged - side
            for n in districts[a]:
                assignment[n] = a
            for n in districts[b]:
                assignment[n] = b
            return True
    return False


def copy_districts(districts):
    return {k: set(s) for k, s in districts.items()}


best = copy_districts(initial)
best_score = score(rep_shares(best))
start = time.time()
for burst in range(N_BURSTS):
    current = copy_districts(best)
    assignment = {n: k for k, s in current.items() for n in s}
    for step in range(BURST_LEN):
        recom_step(current, assignment)
        s = score(rep_shares(current))
        if s >= best_score:
            best = copy_districts(current)
            best_score = s
    if burst % 250 == 0 or burst == N_BURSTS - 1:
        print(f'burst {burst}: {time.time() - start:.0f}s, best score {best_score:.3f}')

burst 0: 0s, best score 5.415


burst 250: 23s, best score 5.438


burst 500: 46s, best score 5.468


burst 750: 70s, best score 5.471


burst 1000: 93s, best score 5.472


burst 1250: 117s, best score 5.472


burst 1500: 142s, best score 5.472


burst 1750: 166s, best score 5.473


burst 2000: 190s, best score 5.505


burst 2250: 216s, best score 6.427


burst 2499: 241s, best score 6.441


In [9]:
TARGET_SEATS = int(best_score)
COUNTY_AWARE = True
land_edges = [(u, v) for u in land for v in land_adj[u] if u < v]


def max_black_share(districts):
    return max(math.fsum(black_vap[n] for n in s) / math.fsum(vap[n] for n in s) for s in districts.values())


def hidden_score(districts):
    # 1) reject maps that lose a safe Republican seat
    if sum(x >= R_THRESHOLD for x in rep_shares(districts).values()) < TARGET_SEATS:
        return float('-inf')
    # 2) until some district is majority-Black, the only goal is raising the highest Black share
    b = max_black_share(districts)
    if b <= BLACK_TARGET:
        return -1e6 + 1e6 * b
    # 3) after that, fewer boundary edges and fewer county pieces is better
    assignment = {n: k for k, s in districts.items() for n in s}
    boundary_edges = sum(assignment[u] != assignment[v] for u, v in land_edges)
    county_pieces = len({(n[:5], assignment[n]) for n in land})
    return -(boundary_edges + COUNTY_WEIGHT * county_pieces)


best_hidden = hidden_score(best)
print('safe Republican seats to keep:', TARGET_SEATS, '| highest Black VAP share: %.3f' % max_black_share(best))
start = time.time()
for burst in range(N_BURSTS_2):
    current = copy_districts(best)
    assignment = {n: k for k, s in current.items() for n in s}
    for step in range(BURST_LEN):
        recom_step(current, assignment)
        s = hidden_score(current)
        if s >= best_hidden:
            best = copy_districts(current)
            best_hidden = s
    if burst % 250 == 0 or burst == N_BURSTS_2 - 1:
        print(f'burst {burst}: {time.time() - start:.0f}s, highest Black VAP share {max_black_share(best):.3f}, best score {best_hidden:.0f}')

safe Republican seats to keep: 6 | highest Black VAP share: 0.398
burst 0: 0s, highest Black VAP share 0.398, best score -601801


burst 250: 26s, highest Black VAP share 0.485, best score -514690


burst 500: 53s, highest Black VAP share 0.502, best score -1328


burst 750: 79s, highest Black VAP share 0.503, best score -1274


burst 1000: 107s, highest Black VAP share 0.500, best score -1235


burst 1250: 137s, highest Black VAP share 0.500, best score -1225


burst 1500: 166s, highest Black VAP share 0.500, best score -1219


burst 1750: 195s, highest Black VAP share 0.500, best score -1217


burst 2000: 221s, highest Black VAP share 0.501, best score -1191


burst 2250: 248s, highest Black VAP share 0.504, best score -1184


burst 2499: 275s, highest Black VAP share 0.508, best score -1181


In [10]:
# number districts 1-12 and attach each water precinct to the district most of its land neighbors are in
order = sorted(best, key=lambda k: min(best[k]))
district_number = {k: i + 1 for i, k in enumerate(order)}
final = {n: district_number[k] for k, s in best.items() for n in s}
for w in water:
    neighbor_districts = [final[q] for q in sorted(adj[w]) if q in final]
    final[w] = max(sorted(set(neighbor_districts)), key=neighbor_districts.count)

nj_unfair_assignment = pd.DataFrame({'GEOID20': nj_precinct_data['GEOID20'],
                                     'District': nj_precinct_data['GEOID20'].map(final)})
nj_unfair_assignment.to_csv('unfair_map_LL.csv', index=False)
print(len(nj_unfair_assignment), 'precincts saved to unfair_map_LL.csv')

6361 precincts saved to unfair_map_LL.csv


## Checking the unfair map

In [11]:
print('Contiguity (course function):')
for i in range(1, K + 1):
    print('District ' + str(i) + ' ' + str(isDistrictContiguous(i, nj_unfair_assignment, nj_contiguity)))

populations = getDistrictPopulations(nj_unfair_assignment, nj_precinct_data, K)
deviations = {i: (p / IDEAL - 1) * 100 for i, p in populations.items()}
print('\nPopulation:', populations)
print('Max deviation from ideal: %.2f%%' % max(abs(x) for x in deviations.values()))

Contiguity (course function):
District 1 True


District 2 True

District 3 True
District 4 True
District 5 True


District 6 True
District 7 True


District 8 True


District 9 True
District 10 True


District 11 True
District 12 True

Population: {1: np.int64(769788), 2: np.int64(772839), 3: np.int64(788631), 4: np.int64(779151), 5: np.int64(778625), 6: np.int64(781150), 7: np.int64(764109), 8: np.int64(786748), 9: np.int64(771307), 10: np.int64(764678), 11: np.int64(771550), 12: np.int64(760418)}
Max deviation from ideal: 1.88%


In [12]:
def district_table(assignment):
    m = assignment.merge(nj_precinct_data.drop(columns='District'), on='GEOID20')
    g = m.groupby('District')
    d, r = g['Dem_2016-2021_Comp'].sum(), g['Rep_2016-2021_Comp'].sum()
    table = pd.DataFrame({
        'Population': g['Total_2020_Total'].sum(),
        'Dev %': (g['Total_2020_Total'].sum() / IDEAL - 1) * 100,
        'Dem %': d / (d + r) * 100,
        'Rep %': r / (d + r) * 100,
        'Black VAP %': g['Black_2020_VAP'].sum() / g['Total_2020_VAP'].sum() * 100,
        'Hispanic VAP %': g['Hispanic_2020_VAP'].sum() / g['Total_2020_VAP'].sum() * 100,
        'Minority VAP %': (1 - g['White_2020_VAP'].sum() / g['Total_2020_VAP'].sum()) * 100,
    })
    table['Winner'] = np.where(r > d, 'R', 'D')
    table['Polsby-Popper'] = [pp_compactness(getDistrictShape(i, assignment, nj_precinct_boundaries)) for i in table.index]
    table['Box Reock'] = [box_reock_compactness(getDistrictShape(i, assignment, nj_precinct_boundaries)) for i in table.index]
    return table, d, r


def map_summary(assignment):
    table, d, r = district_table(assignment)
    total = d + r
    # efficiency gap: wasted votes are all votes for the loser plus winner votes above 50%
    wasted_d = np.where(d > r, d - total / 2, d).sum()
    wasted_r = np.where(r > d, r - total / 2, r).sum()
    counties = assignment.assign(county=assignment['GEOID20'].str[:5]).groupby('county')['District'].nunique()
    summary = {
        'Republican seats': int((table['Winner'] == 'R').sum()),
        'Statewide Rep %': round(r.sum() / total.sum() * 100, 1),
        'Rep seat %': round((table['Winner'] == 'R').mean() * 100, 1),
        'Efficiency gap % (+ favors R)': round((wasted_d - wasted_r) / total.sum() * 100, 1),
        'Median - mean Rep % (+ favors R)': round(table['Rep %'].median() - table['Rep %'].mean(), 2),
        'Mean Polsby-Popper': round(table['Polsby-Popper'].mean(), 3),
        'Min Polsby-Popper': round(table['Polsby-Popper'].min(), 3),
        'Max |pop deviation| %': round(table['Dev %'].abs().max(), 2),
        'Counties split (of 21)': int((counties > 1).sum()),
        'Majority-minority districts (VAP)': int((table['Minority VAP %'] > 50).sum()),
        'Majority-Black districts (VAP)': int((table['Black VAP %'] > 50).sum()),
    }
    return table, summary


unfair_table, unfair_summary = map_summary(nj_unfair_assignment)
current_table, current_summary = map_summary(nj_current_assignment)
print('Unfair map by district:')
display(unfair_table.sort_values('Rep %').round(2))
print('Current map by district:')
display(current_table.sort_values('Rep %').round(2))
pd.DataFrame({'Current map': pd.Series(current_summary, dtype=object),
              'Unfair map': pd.Series(unfair_summary, dtype=object)})

Unfair map by district:


,Population,Dev %,Dem %,Rep %,Black VAP %,Hispanic VAP %,Minority VAP %,Winner,Polsby-Popper,Box Reock
District,,,,,,,,,,
7,764109,-1.29,84.18,15.82,50.83,31.07,84.64,D,0.27,0.57
8,786748,1.64,75.53,24.47,13.39,38.97,70.18,D,0.43,0.56
5,778625,0.59,66.32,33.68,15.52,35.36,58.61,D,0.20,0.52
6,781150,0.91,63.73,36.27,19.78,12.73,40.20,D,0.38,0.60
11,771550,-0.33,63.00,37.00,13.79,24.43,59.12,D,0.22,0.46
4,779151,0.65,59.09,40.91,8.22,21.68,48.81,D,0.48,0.61
9,771307,-0.36,47.89,52.11,7.14,11.48,32.17,R,0.25,0.54
1,769788,-0.55,47.84,52.16,16.33,15.82,37.70,R,0.23,0.51
2,772839,-0.16,47.72,52.28,13.85,11.42,29.38,R,0.43,0.53


Current map by district:


,Population,Dev %,Dem %,Rep %,Black VAP %,Hispanic VAP %,Minority VAP %,Winner,Polsby-Popper,Box Reock
District,,,,,,,,,,
10,746178,-3.60,81.80,18.20,53.40,19.37,79.61,D,0.12,0.35
8,800074,3.36,75.84,24.16,11.73,48.68,73.41,D,0.11,0.37
12,768196,-0.76,65.22,34.78,17.76,19.26,56.59,D,0.16,0.39
1,775340,0.16,60.64,39.36,17.80,12.60,36.86,D,0.42,0.45
9,766863,-0.93,59.92,40.08,11.06,38.91,59.08,D,0.17,0.30
6,778516,0.57,58.42,41.58,12.70,22.01,54.73,D,0.15,0.33
11,769523,-0.59,55.53,44.47,7.65,15.21,35.89,D,0.22,0.56
3,778489,0.57,54.29,45.71,14.18,9.79,34.12,D,0.23,0.38
5,774454,0.05,53.96,46.04,7.02,14.92,38.79,D,0.24,0.36


,Current map,Unfair map
Republican seats,3,6
Statewide Rep %,44.0,44.0
Rep seat %,25.0,50.0
Efficiency gap % (+ favors R),-8.9,18.6
Median - mean Rep % (+ favors R),0.9,4.52
Mean Polsby-Popper,0.211,0.302
Min Polsby-Popper,0.112,0.146
Max |pop deviation| %,3.6,1.88
Counties split (of 21),14,11
Majority-minority districts (VAP),5,4


In [13]:
# robustness: Republican seats under each election in the data, not just the composite
elections = ['2016_Pres', '2017_Gov', '2018_Sen', '2020_Pres', '2020_Sen', '2021_Gov', '2016-2021_Comp']
rows = {}
for e in elections:
    row = {'Statewide Rep %': round(nj_precinct_data['Rep_' + e].sum() / (nj_precinct_data['Rep_' + e].sum() + nj_precinct_data['Dem_' + e].sum()) * 100, 1)}
    for name, a in [('Current map', nj_current_assignment), ('Unfair map', nj_unfair_assignment)]:
        g = a.merge(nj_precinct_data.drop(columns='District'), on='GEOID20').groupby('District')
        row[name + ' R seats'] = int((g['Rep_' + e].sum() > g['Dem_' + e].sum()).sum())
    rows[e] = row
pd.DataFrame.from_dict(rows, orient='index')

,Statewide Rep %,Current map R seats,Unfair map R seats
2016_Pres,42.7,3,6
2017_Gov,42.8,2,5
2018_Sen,44.2,3,6
2020_Pres,41.9,2,3
2020_Sen,41.7,2,3
2021_Gov,48.4,3,6
2016-2021_Comp,44.0,3,6
